# ⚽ Premier League Match Predictor (VERSIÓN RÁPIDA)
## Análisis de Datos y Predicción con Machine Learning

**Versión optimizada**: Ejecuta en ~3-5 minutos

---

In [ ]:
# 1. INSTALACIÓN RÁPIDA
!pip install -q pandas numpy matplotlib seaborn scikit-learn requests

print("✅ Dependencias instaladas")

In [ ]:
# 2. IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
import requests
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Crear directorios
os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("✅ Librerías importadas y directorios creados")

In [ ]:
# 3. OBTENCIÓN DE DATOS
API_KEY = "f50c3bb69922405b8963e15c66c23877"
LEAGUE = "PL"
API_BASE = "https://api.football-data.org/v4"
HEADERS = {"X-Auth-Token": API_KEY}

url = f"{API_BASE}/competitions/{LEAGUE}/standings"
print("📡 Obteniendo datos de la API...")

try:
    response = requests.get(url, headers=HEADERS, timeout=10)
    data = response.json()
    table = data["standings"][0]["table"]
    
    teams = []
    for row in table:
        teams.append({
            "id": row["team"]["id"],
            "name": row["team"]["name"],
            "position": row["position"],
            "played": row["playedGames"],
            "won": row["won"],
            "draw": row["draw"],
            "lost": row["lost"],
            "points": row["points"],
            "goalsFor": row["goalsFor"],
            "goalsAgainst": row["goalsAgainst"],
            "goalDifference": row["goalDifference"]
        })
    
    df_teams = pd.DataFrame(teams)
    print(f"✅ Datos obtenidos: {len(df_teams)} equipos")
except:
    print("⚠️ Error de API, usando datos de ejemplo...")
    # Datos de ejemplo por si falla la API
    df_teams = pd.DataFrame({
        'id': range(1, 21),
        'name': [f'Team {i}' for i in range(1, 21)],
        'position': range(1, 21),
        'played': [10] * 20,
        'won': [5, 4, 4, 3, 3, 3, 2, 2, 2, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        'draw': [2] * 20,
        'lost': [3, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8],
        'points': [17, 14, 14, 11, 11, 11, 8, 8, 8, 5, 5, 5, 2, 2, 2, 2, 2, 2, 2, 2],
        'goalsFor': [15, 12, 12, 10, 10, 10, 8, 8, 8, 5, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3],
        'goalsAgainst': [8, 10, 10, 12, 12, 12, 14, 14, 14, 16, 16, 16, 18, 18, 18, 18, 18, 18, 18, 18],
        'goalDifference': [7, 2, 2, -2, -2, -2, -6, -6, -6, -11, -11, -11, -15, -15, -15, -15, -15, -15, -15, -15]
    })

df_teams.to_csv('data/teams_raw.csv', index=False)
print("\nPrimeros 5 equipos:")
df_teams.head()

In [ ]:
# 4. FEATURE ENGINEERING
print("⚙️ Creando features...")

df_features = df_teams.copy()

df_features['points_per_game'] = df_features['points'] / df_features['played'].replace(0, 1)
df_features['goals_for_per_game'] = df_features['goalsFor'] / df_features['played'].replace(0, 1)
df_features['goals_against_per_game'] = df_features['goalsAgainst'] / df_features['played'].replace(0, 1)
df_features['attack_strength'] = df_features['goals_for_per_game']
df_features['defense_strength'] = 1 / (df_features['goals_against_per_game'].replace(0, 0.1))
df_features['win_rate'] = (df_features['won'] / df_features['played'].replace(0, 1)) * 100
df_features['goal_difference_per_game'] = df_features['goalDifference'] / df_features['played'].replace(0, 1)
df_features['form_score'] = df_features['points_per_game'] * 0.6 + df_features['goal_difference_per_game'] * 0.4
df_features['team_score'] = (
    df_features['attack_strength'] * 0.35 +
    df_features['defense_strength'] * 0.35 +
    df_features['form_score'] * 0.30
)

df_features.to_csv('data/teams_with_features.csv', index=False)

print("✅ Features creados")
print("\nTop 5 equipos por score:")
df_features[['name', 'team_score', 'attack_strength', 'defense_strength']].sort_values('team_score', ascending=False).head()

In [ ]:
# 5. VISUALIZACIÓN RÁPIDA (Solo 3 gráficos esenciales)
print("📊 Generando visualizaciones...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Puntos
df_sorted = df_features.sort_values('points', ascending=True)
axes[0, 0].barh(df_sorted['name'], df_sorted['points'], color='steelblue')
axes[0, 0].set_xlabel('Puntos')
axes[0, 0].set_title('Puntos por Equipo', fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Ataque
top_attack = df_features.nlargest(10, 'goalsFor').sort_values('goalsFor')
axes[0, 1].barh(top_attack['name'], top_attack['goalsFor'], color='green', alpha=0.7)
axes[0, 1].set_xlabel('Goles a Favor')
axes[0, 1].set_title('Top 10 Ataques', fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Correlación
numeric_cols = ['points', 'goalsFor', 'goalsAgainst', 'won', 'draw', 'lost']
corr = df_features[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 0])
axes[1, 0].set_title('Matriz de Correlación', fontweight='bold')

# 4. Team Scores
df_sorted = df_features.sort_values('team_score', ascending=True)
axes[1, 1].barh(df_sorted['name'], df_sorted['team_score'], color='purple', alpha=0.7)
axes[1, 1].set_xlabel('Team Score')
axes[1, 1].set_title('Score Total por Equipo', fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/visualizaciones_principales.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualizaciones guardadas")

In [ ]:
# 6. DATASET PARA ML (RÁPIDO)
print("🤖 Creando dataset de entrenamiento...")

matches = []
for i, team_a in df_features.iterrows():
    for j, team_b in df_features.iterrows():
        if i != j:
            match = {
                'teamA_id': team_a['id'],
                'teamB_id': team_b['id'],
                'teamA_name': team_a['name'],
                'teamB_name': team_b['name'],
                'attack_diff': team_a['attack_strength'] - team_b['attack_strength'],
                'defense_diff': team_a['defense_strength'] - team_b['defense_strength'],
                'form_diff': team_a['form_score'] - team_b['form_score'],
                'points_diff': team_a['points'] - team_b['points'],
                'score_diff': team_a['team_score'] - team_b['team_score'],
            }
            prob_a = 1 / (1 + np.exp(-match['score_diff']))
            match['winner'] = 1 if prob_a > 0.5 else 0
            matches.append(match)

df_matches = pd.DataFrame(matches)
df_matches.to_csv('data/matches_dataset.csv', index=False)

print(f"✅ Dataset: {len(df_matches)} enfrentamientos")

In [ ]:
# 7. ENTRENAMIENTO RÁPIDO (Solo Random Forest)
print("🧠 Entrenando modelo Random Forest...")

feature_columns = ['attack_diff', 'defense_diff', 'form_diff', 'points_diff', 'score_diff']
X = df_matches[feature_columns]
y = df_matches['winner']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Solo Random Forest (el mejor)
rf_model = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"✅ Modelo entrenado - Accuracy: {accuracy:.4f}")

# Guardar modelo
with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("💾 Modelo guardado")

In [ ]:
# 8. EVALUACIÓN RÁPIDA
print("📈 Evaluando modelo...")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=['Team B', 'Team A'],
           yticklabels=['Team B', 'Team A'])
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de Confusión')

# ROC Curve
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/evaluacion_modelo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ AUC Score: {roc_auc:.4f}")

In [ ]:
# 9. SISTEMA DE PREDICCIÓN
print("🎯 Sistema de predicción listo\n")

def predict_match(team_a_id, team_b_id):
    team_a = df_features[df_features['id'] == team_a_id].iloc[0]
    team_b = df_features[df_features['id'] == team_b_id].iloc[0]
    
    match = pd.DataFrame([{
        'attack_diff': team_a['attack_strength'] - team_b['attack_strength'],
        'defense_diff': team_a['defense_strength'] - team_b['defense_strength'],
        'form_diff': team_a['form_score'] - team_b['form_score'],
        'points_diff': team_a['points'] - team_b['points'],
        'score_diff': team_a['team_score'] - team_b['team_score'],
    }])
    
    proba = rf_model.predict_proba(match)[0]
    prob_b, prob_a = proba[0] * 100, proba[1] * 100
    
    return {
        'teamA': team_a['name'],
        'teamB': team_b['name'],
        'probA': round(prob_a, 2),
        'probB': round(prob_b, 2),
        'winner': team_a['name'] if prob_a > prob_b else team_b['name'],
        'confidence': round(max(prob_a, prob_b), 2)
    }

# Ejemplo
example = predict_match(df_features.iloc[0]['id'], df_features.iloc[1]['id'])
print(f"🧪 Ejemplo: {example['teamA']} vs {example['teamB']}")
print(f"   Ganador: {example['winner']} ({example['confidence']}%)")
print(f"   Probabilidades: {example['probA']}% - {example['probB']}%")

In [ ]:
# 10. EXPORTAR CSVs RÁPIDO
print("💾 Exportando CSVs...\n")

# 1. Teams
output_teams = df_features[['id', 'name', 'position', 'points', 'goalsFor', 'goalsAgainst',
                           'attack_strength', 'defense_strength', 'team_score']]
output_teams.to_csv('outputs/teams_analysis.csv', index=False)
print("✅ teams_analysis.csv")

# 2. Predicciones (20 ejemplos)
predictions = []
for i in range(min(5, len(df_features))):
    for j in range(i+1, min(5, len(df_features))):
        pred = predict_match(df_features.iloc[i]['id'], df_features.iloc[j]['id'])
        predictions.append(pred)

pd.DataFrame(predictions).to_csv('outputs/match_predictions.csv', index=False)
print(f"✅ match_predictions.csv ({len(predictions)} predicciones)")

# 3. Resumen
summary = pd.DataFrame({
    'metric': ['Total Teams', 'Best Model', 'Accuracy', 'AUC Score', 'Features'],
    'value': [len(df_features), 'Random Forest', f"{accuracy:.4f}", f"{roc_auc:.4f}", len(feature_columns)]
})
summary.to_csv('outputs/project_summary.csv', index=False)
print("✅ project_summary.csv")

print("\n🎉 ¡ANÁLISIS COMPLETADO!")

In [ ]:
# 11. RESUMEN FINAL
print("="*60)
print(" "*20 + "RESUMEN FINAL")
print("="*60)
print(f"\n🏆 Equipos analizados: {len(df_features)}")
print(f"🤖 Modelo: Random Forest")
print(f"🎯 Accuracy: {accuracy:.4f}")
print(f"📈 AUC Score: {roc_auc:.4f}")
print(f"📊 Visualizaciones: 2 gráficos")
print(f"💾 CSVs exportados: 3 archivos")
print(f"\n📁 Archivos en: data/, outputs/, models/")
print("\n" + "="*60)